Battery Returns - Bronze, Silver, Gold, and validated Eventhouse publication


In [ ]:
stage = 'bronze'
execution_id = 'manual'


In [ ]:
"""Code deployed to a new Fabric notebook; each pipeline activity selects a stage."""

import hashlib
import json
import pathlib
from datetime import datetime, timezone

import requests
from notebookutils import mssparkutils
from pyspark.sql import Window, functions as F, types as T

spark.conf.set("spark.sql.session.timeZone", "UTC")
expected = json.loads(mssparkutils.fs.head("Files/audit/expected.json", 100000))
counts = expected["expected"]
columns = expected["columns"]
event_fields = expected["event_fields"]
run_id = execution_id
audit_root = "Files/audit/runs/" + run_id
mssparkutils.fs.mkdirs(audit_root)
for schema in ("bronze", "silver", "gold"):
    spark.sql("CREATE SCHEMA IF NOT EXISTS " + schema)

ID_FIELDS = {
    "data_label", "battery_serial", "product_id", "batch_id", "shipment_id",
    "factory_id", "retailer_id", "distributor_id", "return_id",
    "originator_id", "submitted_by_id", "receiving_warehouse_id",
}
DATE_FIELDS = {
    "manufactured_at", "invoice_at", "shipped_at", "delivered_at",
    "event_time", "return_registered_at", "warranty_expires_at",
}
INTEGER_FIELDS = {"revision", "warranty_months"}


def write_table(frame, name):
    frame.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)


def audit(name, result):
    result = {
        "stage": name, "pipeline_run_id": run_id, "data_label": "SYNTHETIC",
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
        **result,
    }
    mssparkutils.fs.put(audit_root + "/" + name + ".json", json.dumps(result, indent=2), True)
    mssparkutils.fs.put("Files/audit/latest_" + name + ".json", json.dumps(result, indent=2), True)
    print(json.dumps(result, indent=2))
    return result


def read_audit(name):
    return json.loads(mssparkutils.fs.head(audit_root + "/" + name + ".json", 100000))


def raw_schema(fields):
    return T.StructType([T.StructField(name, T.StringType(), True) for name in fields])


def normalize(frame, fields):
    frame = frame.select(
        *[F.coalesce(F.trim(F.col(name)), F.lit("")).alias(name) for name in fields],
        *[F.col(name) for name in frame.columns if name.startswith("_")],
    )
    for name in fields:
        if name in ID_FIELDS:
            frame = frame.withColumn(name, F.upper(name))
        if name == "originator_id":
            # The synthetic customer identifier uses an uppercase prefix and lowercase hash suffix.
            pattern = r"^(SYN-CUST-[A-Z]{2}-)([0-9A-F]{9})$"
            frame = frame.withColumn(name, F.when(
                F.col(name).rlike(pattern),
                F.concat(
                    F.regexp_extract(name, pattern, 1),
                    F.lower(F.regexp_extract(name, pattern, 2)),
                ),
            ).otherwise(F.col(name)))
        if name in DATE_FIELDS:
            frame = frame.withColumn(name, F.expr("try_to_timestamp(" + name + ")"))
        if name in INTEGER_FIELDS:
            frame = frame.withColumn(name, F.expr("try_cast(" + name + " as int)"))
        if name in expected["category_values"]:
            mapping = expected["category_values"][name]
            canonical = F.create_map(*[
                value for label in mapping for value in (F.lit(label.lower()), F.lit(label))
            ])
            lowered = F.lower(name)
            if name == "originator_type":
                lowered = F.when(lowered.isin("not provided", "unknown", "n/a"), F.lit("unknown")).otherwise(lowered)
            frame = frame.withColumn(name, F.coalesce(canonical[lowered], F.col(name)))
    return frame


def assert_required(frame, fields, label):
    invalid = F.lit(False)
    for name in fields:
        invalid = invalid | F.col(name).isNull()
        if name not in DATE_FIELDS | INTEGER_FIELDS:
            invalid = invalid | (F.length(name) == 0)
    assert frame.filter(invalid).limit(1).count() == 0, label + ": invalid required values"


def canonical_rows(frame):
    expressions = []
    for name in columns:
        value = F.col(name)
        if name in DATE_FIELDS:
            value = F.date_format(value, "yyyy-MM-dd'T'HH:mm:ss'Z'")
        else:
            value = value.cast("string")
        expressions.append(F.coalesce(value, F.lit("")).alias(name))
    return [row.asDict() for row in frame.select(*expressions).orderBy("return_id").collect()]


def digest(rows):
    text = "\n".join(
        json.dumps([row[name] for name in columns], ensure_ascii=True, separators=(",", ":"))
        for row in rows
    )
    return hashlib.sha256(text.encode()).hexdigest()


def load_bronze():
    actual_files = {
        entry.name.rstrip("/") for entry in mssparkutils.fs.ls("Files/landing")
        if not entry.isDir
    }
    assert actual_files == set(expected["source_files"]), "Landing must contain exactly the four declared source files"
    for filename, details in expected["source_files"].items():
        path = pathlib.Path("/lakehouse/default/Files/landing") / filename
        sha = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
                sha.update(block)
        assert sha.hexdigest() == details["sha256"], "Source checksum mismatch: " + filename
    loaded = {}
    for source, fields in (
        ("products", expected["product_fields"]),
        ("batteries", expected["battery_fields"]),
        ("shipments", expected["shipment_fields"]),
        ("return_events", event_fields),
    ):
        filename = source + (".json" if source == "return_events" else ".csv")
        path = "Files/landing/" + filename
        if source == "return_events":
            schema = T.StructType([T.StructField("events", T.ArrayType(raw_schema(fields)))])
            frame = spark.read.schema(schema).option("multiLine", True).option("mode", "FAILFAST").json(path)
            frame = frame.select(F.explode("events").alias("event")).select("event.*")
        else:
            frame = spark.read.schema(raw_schema(fields)).option("header", True).option("mode", "FAILFAST").csv(path)
        frame = (
            frame.withColumn("_source_file", F.lit(filename))
            .withColumn("_source_sha256", F.lit(expected["source_files"][filename]["sha256"]))
            .withColumn("_pipeline_run_id", F.lit(run_id))
            .withColumn("_ingested_at", F.current_timestamp())
        )
        write_table(frame, "bronze." + source)
        loaded[source] = spark.table("bronze." + source).count()
    assert loaded == {
        "products": counts["products"], "batteries": counts["batteries"],
        "shipments": counts["shipments"], "return_events": counts["bronze_events"],
    }, loaded
    return audit("bronze", {"rows": loaded, "source_checksums_verified": True})


def transform_silver():
    read_audit("bronze")
    dimensions = {}
    for source, fields, key in (
        ("products", expected["product_fields"], "product_id"),
        ("shipments", expected["shipment_fields"], "shipment_id"),
        ("batteries", expected["battery_fields"], "battery_serial"),
    ):
        frame = normalize(spark.table("bronze." + source), fields)
        assert_required(frame, fields, source)
        assert frame.filter(F.col("data_label") != "SYNTHETIC").limit(1).count() == 0
        assert frame.groupBy(key).count().filter("count > 1").limit(1).count() == 0, source + ": duplicate master key"
        write_table(frame, "silver." + source)
        dimensions[source] = spark.table("silver." + source)
    products, shipments, batteries = (
        dimensions["products"], dimensions["shipments"], dimensions["batteries"]
    )
    assert products.filter("warranty_months <= 0").limit(1).count() == 0
    assert shipments.join(products, "product_id", "left_anti").limit(1).count() == 0
    assert batteries.join(products, "product_id", "left_anti").limit(1).count() == 0
    assert batteries.join(shipments, ["shipment_id", "product_id", "batch_id"], "left_anti").limit(1).count() == 0
    events = normalize(spark.table("bronze.return_events"), event_fields)
    events = events.join(
        batteries.select("battery_serial").withColumn("_battery_exists", F.lit(True)),
        "battery_serial", "left",
    )
    valid_status = {
        "ReturnRegistered": ("Requested", 1), "ReturnReceived": ("Received", 2),
        "InspectionCompleted": ("Inspected", 3), "ReturnClosed": ("Closed", 4),
    }
    lifecycle_valid = F.lit(False)
    for event_type, (status, revision) in valid_status.items():
        lifecycle_valid = lifecycle_valid | (
            (F.col("event_type") == event_type)
            & (F.col("return_status") == status) & (F.col("revision") == revision)
        )
    required_keys = [
        "demo_run_id", "event_id", "return_id", "battery_serial", "submitted_by_id",
        "receiving_warehouse_id",
    ]
    missing_key = F.lit(False)
    for key in required_keys:
        missing_key = missing_key | (F.length(key) == 0)
    events = events.withColumn("_rejection_reason", F.concat_ws("; ",
        F.when(missing_key, "Missing required key"),
        F.when(F.col("event_time").isNull() | F.col("return_registered_at").isNull(), "Invalid timestamp"),
        F.when((F.length("battery_serial") > 0) & F.col("_battery_exists").isNull(), "Unknown battery foreign key"),
        F.when(F.col("revision").isNull() | (F.col("revision") <= 0), "Invalid revision"),
        F.when(F.col("data_label") != "SYNTHETIC", "Unexpected data label"),
        F.when(F.col("demo_run_id") != expected["dataset_id"], "Unexpected dataset"),
        F.when(F.col("event_time") < F.col("return_registered_at"), "Event precedes registration"),
        F.when(F.col("event_time") > F.to_timestamp(F.lit(expected["as_of_utc"])), "Event after dataset cutoff"),
        F.when(~F.col("originator_type").isin(expected["category_values"]["originator_type"]), "Invalid originator type"),
        F.when((F.col("originator_type") == "Unknown") != (F.length("originator_id") == 0), "Inconsistent unknown initiator"),
        F.when(~F.col("submitted_by_type").isin(expected["category_values"]["submitted_by_type"]), "Invalid submitter type"),
        F.when(~F.coalesce(lifecycle_valid, F.lit(False)), "Inconsistent lifecycle stage"),
    )).drop("_battery_exists")
    rejected = events.filter(F.col("_rejection_reason") != "").withColumn("_quarantined_at", F.current_timestamp())
    write_table(rejected, "silver.quarantine_events")
    valid = events.filter(F.col("_rejection_reason") == "").drop("_rejection_reason")
    payload_hash = F.sha2(F.to_json(F.struct(*event_fields)), 256)
    conflicts = valid.groupBy("event_id").agg(F.countDistinct(payload_hash).alias("versions")).filter("versions > 1")
    assert conflicts.limit(1).count() == 0, "Conflicting payloads with the same event ID"
    deduped = valid.dropDuplicates(["event_id"])
    assert deduped.groupBy("demo_run_id", "return_id", "revision").count().filter("count > 1").limit(1).count() == 0
    write_table(deduped, "silver.return_events")
    raw_count = spark.table("bronze.return_events").count()
    quarantined = spark.table("silver.quarantine_events").count()
    unique_count = spark.table("silver.return_events").count()
    replays = raw_count - quarantined - unique_count
    assert quarantined == counts["quarantined_events"]
    assert unique_count == counts["silver_events"]
    assert replays == counts["exact_replays_removed"]
    return audit("silver", {
        "bronze_events": raw_count, "quarantined_events": quarantined,
        "valid_raw_events": raw_count - quarantined, "silver_events": unique_count,
        "exact_replays_removed": replays,
        "quarantine_reasons": [row.asDict() for row in rejected.groupBy("_rejection_reason").count().collect()],
    })


def build_gold():
    read_audit("silver")
    events = spark.table("silver.return_events").alias("e")
    batteries = spark.table("silver.batteries").alias("b")
    shipments = spark.table("silver.shipments").alias("s")
    products = spark.table("silver.products").alias("p")
    joined = (
        events.join(batteries, F.col("e.battery_serial") == F.col("b.battery_serial"))
        .join(shipments, F.col("b.shipment_id") == F.col("s.shipment_id"))
        .join(products, F.col("b.product_id") == F.col("p.product_id"))
    )
    field_sources = {name: "e" for name in event_fields}
    field_sources.update({name: "b" for name in expected["battery_fields"] if name != "data_label"})
    field_sources.update({name: "s" for name in (
        "country", "shipped_at", "delivered_at", "distributor_id",
    )})
    field_sources.update({name: "p" for name in (
        "product_name", "battery_technology", "warranty_months",
    )})
    enriched = joined.select(*[
        F.col(field_sources[name] + "." + name).alias(name)
        for name in columns if name not in {"warranty_expires_at", "warranty_status"}
    ])
    enriched = enriched.withColumn(
        "warranty_expires_at", F.expr("timestampadd(MONTH, warranty_months, invoice_at)")
    ).withColumn(
        "warranty_status",
        F.when(F.col("return_registered_at") <= F.col("warranty_expires_at"), "InWarranty").otherwise("OutOfWarranty"),
    ).select(*columns)
    assert enriched.count() == counts["silver_events"], "Dimension joins lost or multiplied event rows"
    assert enriched.filter(
        (F.col("manufactured_at") > F.col("shipped_at"))
        | (F.col("shipped_at") > F.col("delivered_at"))
        | (F.col("delivered_at") > F.col("invoice_at"))
        | (F.col("invoice_at") > F.col("return_registered_at"))
        | (F.col("return_registered_at") > F.col("event_time"))
    ).limit(1).count() == 0, "Invalid business date sequence"
    window = Window.partitionBy("demo_run_id", "return_id").orderBy(
        F.col("revision").desc(), F.col("event_time").desc(), F.col("event_id").desc()
    )
    latest = enriched.withColumn("_rank", F.row_number().over(window)).filter("_rank = 1").drop("_rank")
    rows = canonical_rows(latest)
    gold_digest = digest(rows)
    assert len(rows) == counts["gold_cases"]
    assert gold_digest == counts["gold_sha256"], "Gold differs from the canonical baseline case values"
    assert sum(row["originator_type"] == "Unknown" for row in rows) == counts["unknown_origin_cases"]
    write_table(enriched, "gold.return_event_history")
    write_table(latest, "gold.return_cases")
    cases = spark.table("gold.return_cases")
    write_table(
        cases.groupBy("batch_id", "product_id", "country").agg(
            F.count("*").alias("return_cases"), F.countDistinct("battery_serial").alias("distinct_batteries"),
            F.min("return_registered_at").alias("first_return"), F.max("return_registered_at").alias("last_return"),
        ), "gold.batch_summary",
    )
    write_table(
        cases.withColumn("return_month", F.date_format("return_registered_at", "yyyy-MM"))
        .groupBy("return_month", "country").agg(
            F.count("*").alias("return_cases"),
            F.sum(F.when(F.col("originator_type") == "Unknown", 1).otherwise(0)).alias("unknown_initiator_cases"),
            F.sum(F.when(F.col("warranty_status") == "InWarranty", 1).otherwise(0)).alias("in_warranty_cases"),
        ), "gold.monthly_summary",
    )
    write_table(
        cases.groupBy("originator_type", "submitted_by_type").agg(F.count("*").alias("return_cases")),
        "gold.initiator_summary",
    )
    return audit("gold", {
        "gold_cases": len(rows), "gold_event_history": enriched.count(),
        "gold_sha256": gold_digest, "all_case_values_match_baseline": True,
        "unknown_origin_cases": counts["unknown_origin_cases"],
    })


CLUSTER = "https://trd-x6s8dxpyykghc2cybq.z6.kusto.fabric.microsoft.com"
DATABASE_ID = "c688ad4f-786d-46a3-a0b7-400177d2871f"


def kusto(query, management=False):
    # Tokens stay in notebook memory and are sent only to this verified Eventhouse.
    token = mssparkutils.credentials.getToken("kusto")
    response = requests.post(
        CLUSTER + "/v1/rest/" + ("mgmt" if management else "query"),
        headers={"Authorization": "Bearer " + token},
        json={"db": DATABASE_ID, "csl": query}, timeout=120,
    )
    if not response.ok:
        raise RuntimeError(f"Eventhouse HTTP {response.status_code}: {response.text[:1200]}")
    result = response.json()
    table = result["Tables"][0]
    names = [column["ColumnName"] for column in table["Columns"]]
    return [dict(zip(names, row)) for row in table["Rows"]]


def ingest(table, rows, tag):
    tag_literal = json.dumps([tag])
    command = (
        f".ingest inline into table {table} with(format='multijson', "
        f"tags='{tag_literal}', ingestIfNotExists='{tag_literal}') <|\n"
        + "\n".join(json.dumps(row, separators=(",", ":")) for row in rows)
    )
    kusto(command, True)


def publish_gold():
    gold_check = read_audit("gold")
    silver_check = read_audit("silver")
    read_audit("bronze")
    rows = canonical_rows(spark.table("gold.return_cases"))
    assert digest(rows) == gold_check["gold_sha256"] == counts["gold_sha256"]
    source_digest = hashlib.sha256(
        json.dumps(expected["source_files"], sort_keys=True, separators=(",", ":")).encode()
    ).hexdigest()
    publish_id = "gold-" + hashlib.sha256((source_digest + counts["gold_sha256"]).encode()).hexdigest()[:24]
    existing = kusto(
        f"ReturnsGoldCommits | where publish_id == '{publish_id}' | summarize Commits=count()"
    )[0]["Commits"]
    if not existing:
        for offset in range(0, len(rows), 400):
            batch = []
            for row in rows[offset:offset + 400]:
                row = dict(row)
                for name in INTEGER_FIELDS:
                    row[name] = int(row[name])
                row["publish_id"] = publish_id
                batch.append(row)
            ingest("ReturnsGoldRaw", batch, f"ingest-by:gold-{publish_id}-{offset:05d}")
            print(f"Eventhouse Gold publication: {min(offset + 400, len(rows))}/{len(rows)}", flush=True)
    cloud = kusto(
        f"ReturnsGoldRaw | where publish_id == '{publish_id}'"
        " | summarize GoldCases=count(), DistinctCases=count_distinct(return_id),"
        " UnknownOrigins=countif(originator_type == 'Unknown')"
    )[0]
    assert cloud == {
        "GoldCases": counts["gold_cases"], "DistinctCases": counts["gold_cases"],
        "UnknownOrigins": counts["unknown_origin_cases"],
    }, cloud
    comparisons = " and ".join(f"g_{name} == h_{name}" for name in columns)
    golden_projection = ", ".join(f"g_{name}={name}" for name in columns)
    history_projection = ", ".join(f"h_{name}={name}" for name in columns)
    comparison = kusto(
        f"let G = ReturnsGoldRaw | where publish_id == '{publish_id}' | project {golden_projection};"
        f"let H = ReturnsHistoricalLatest() | where demo_run_id == '{expected['dataset_id']}'"
        f" | project {history_projection};"
        "G | join kind=fullouter H on $left.g_return_id == $right.h_return_id"
        f" | summarize ComparedCases=count(), Mismatches=countif(not({comparisons}))"
    )[0]
    assert comparison == {"ComparedCases": counts["gold_cases"], "Mismatches": 0}, comparison
    if not existing:
        commit = {
            "publish_id": publish_id, "published_at": datetime.now(timezone.utc).isoformat(),
            "data_label": "SYNTHETIC", "dataset_id": expected["dataset_id"],
            "bronze_events": silver_check["bronze_events"],
            "quarantined_events": silver_check["quarantined_events"],
            "valid_raw_events": silver_check["valid_raw_events"],
            "silver_events": silver_check["silver_events"],
            "replays_removed": silver_check["exact_replays_removed"],
            "gold_cases": len(rows), "gold_sha256": counts["gold_sha256"],
            "pipeline_run_id": run_id, "source_manifest_sha256": source_digest,
        }
        ingest("ReturnsGoldCommits", [commit], f"ingest-by:gold-commit-{publish_id}")
    served = kusto("ReturnsGoldLatest() | summarize GoldCases=count()")[0]["GoldCases"]
    assert served == counts["gold_cases"]
    result = audit("publish", {
        "publish_id": publish_id, "gold_cases": served,
        "new_publication": not bool(existing), "reconciliation": comparison,
        "live_tables_untouched": True,
    })
    audit_frame = spark.createDataFrame([(
        run_id, expected["dataset_id"], publish_id, counts["gold_sha256"],
        counts["bronze_events"], counts["quarantined_events"], counts["silver_events"],
        counts["exact_replays_removed"], counts["gold_cases"], datetime.now(timezone.utc).replace(tzinfo=None),
    )], "pipeline_run_id string, dataset_id string, publish_id string, gold_sha256 string,"
       "bronze_events long, quarantined_events long, silver_events long, replays_removed long,"
       "gold_cases long, completed_at timestamp")
    if spark.catalog.tableExists("gold.pipeline_runs"):
        from delta.tables import DeltaTable
        (
            DeltaTable.forName(spark, "gold.pipeline_runs").alias("t")
            .merge(audit_frame.alias("s"), "t.pipeline_run_id = s.pipeline_run_id")
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
        )
    else:
        write_table(audit_frame, "gold.pipeline_runs")
    return result


stages = {
    "bronze": load_bronze, "silver": transform_silver,
    "gold": build_gold, "publish": publish_gold,
}
assert stage in stages, "Unknown medallion stage"
result = stages[stage]()
mssparkutils.notebook.exit(json.dumps(result))
